In [ ]:
import spacy
import yake
from keybert import KeyBERT
import json
import pandas as pd
from collections import defaultdict
import re
from collections import Counter

class NutritionNER:
    def __init__(self):
        # Load spaCy model
        self.nlp = spacy.load("en_core_web_sm")
        
        # Initialize YAKE
        self.yake_extractor = yake.KeywordExtractor(
            lan="en", 
            n=3,
            dedupLim=0.9,
            top=20
        )
        
        # Initialize KeyBERT
        self.keybert_model = KeyBERT()
        
        # Ontology patterns
        self.ontology_patterns = {
            'ingredient': [
                r'\b(?:whole grain|rye bread|oatmeal|barley|salmon|trout|herring|'
                r'beans|lentils|peas|soy|tofu|nuts|seeds|vegetables|fruits|berries|'
                r'potatoes|dairy|milk|cheese|yogurt|eggs|meat|poultry|fish)\b',
                r'\b(?:[A-Z][a-z]+ (?:oil|bread|milk|cheese|yogurt|fish|meat))\b'
            ],
            'nutrient': [
                r'\b(?:vitamin [A-Z]|[A-Z] vitamins?|minerals?|iron|zinc|calcium|'
                r'selenium|iodine|fibre|fiber|protein|carbohydrates?|fat|sugar|'
                r'saturated fat|unsaturated fat|omega-3|omega-6|folate|'
                r'vitamin C|vitamin D|vitamin E|vitamin B12|vitamin B6)\b'
            ],
            'technique': [
                r'\b(?:cooking|boiling|steaming|frying|baking|roasting|grilling|'
                r'fermentation|soaking|germination|processing|preparation|'
                r'filtered|unfiltered|fortification)\b'
            ],
            'healthOutcome': [
                r'\b(?:cardiovascular disease|diabetes|obesity|cancer|'
                r'colorectal cancer|osteoporosis|anaemia|anemia|hypertension|'
                r'high blood pressure|stroke|dental caries|tooth decay|'
                r'metabolic syndrome|inflammation)\b'
            ],
            'environmentImpact': [
                r'\b(?:climate impact|carbon footprint|greenhouse gas|'
                r'biodiversity loss|eutrophication|water footprint|'
                r'land use|resource use|environmental impact)\b'
            ]
        }

    def extract_entities_spacy(self, text):
        """Extract entities using spaCy's built-in NER"""
        doc = self.nlp(text)
        return [{'text': ent.text, 'label': ent.label_, 'start': ent.start_char, 'end': ent.end_char} for ent in doc.ents]

    def extract_keyphrases_yake(self, text):
        """Extract keyphrases using YAKE"""
        return [kw[0] for kw in self.yake_extractor.extract_keywords(text)]

    def extract_keyphrases_keybert(self, text):
        """Extract keyphrases using KeyBERT"""
        return [kw[0] for kw in self.keybert_model.extract_keywords(
            text,
            keyphrase_ngram_range=(1, 3),
            stop_words='english',
            top_n=10
        )]

    def map_to_ontology(self, text, keyphrases):
        """Map extracted phrases to ontology classes"""
        mapped_entities = defaultdict(list)
        for phrase in keyphrases:
            for category, patterns in self.ontology_patterns.items():
                for pattern in patterns:
                    if re.search(pattern, phrase, re.IGNORECASE):
                        mapped_entities[category].append(phrase)
                        break
        return dict(mapped_entities)

    def resolve_coreferences(self, entities):
        """Resolve coreferences for common nutrition terms"""
        coreference_map = {
            'vitamin c': ['ascorbic acid', 'vit c'],
            'vitamin d': ['vit d', 'cholecalciferol'],
            'vitamin e': ['vit e', 'tocopherol'],
            'vitamin b12': ['vitamin b12', 'cobalamin', 'b12'],
            'fiber': ['fibre', 'dietary fiber', 'dietary fibre'],
            'saturated fat': ['saturated fatty acids', 'solid fat'],
            'unsaturated fat': ['unsaturated fatty acids', 'liquid fat'],
            'cardiovascular disease': ['cvd', 'heart disease'],
            'type 2 diabetes': ['t2d', 'adult-onset diabetes']
        }

        resolved_entities = defaultdict(list)
        all_phrases = [phrase for phrases in entities.values() for phrase in phrases]

        for canonical, variants in coreference_map.items():
            for phrase in all_phrases:
                lower_phrase = phrase.lower()
                if any(variant in lower_phrase for variant in [canonical] + variants):
                    for category, phrases in entities.items():
                        if phrase in phrases:
                            resolved_entities[category].append(canonical)
                            break

        # Add non-coreferenced
        for category, phrases in entities.items():
            for phrase in phrases:
                lower_phrase = phrase.lower()
                if not any(lower_phrase in variants or canonical in lower_phrase 
                           for canonical, variants in coreference_map.items()):
                    resolved_entities[category].append(phrase)

        for category in resolved_entities:
            resolved_entities[category] = list(set(resolved_entities[category]))

        return dict(resolved_entities)

    def process_text_chunk(self, chunk):
        """Process a single text chunk"""
        text = chunk.get('text', '')
        metadata = chunk.get('metadata', {})

        spacy_entities = self.extract_entities_spacy(text)
        yake_keyphrases = self.extract_keyphrases_yake(text)
        keybert_keyphrases = self.extract_keyphrases_keybert(text)

        all_keyphrases = list(set(yake_keyphrases + keybert_keyphrases))

        mapped_entities = self.map_to_ontology(text, all_keyphrases)
        resolved_entities = self.resolve_coreferences(mapped_entities)

        return {
            'chunk_id': metadata.get('chunk_id', ''),
            'page': metadata.get('page', ''),
            'section': metadata.get('section', ''),
            'text_preview': text[:100] + '...' if len(text) > 100 else text,
            'spacy_entities': spacy_entities,
            'keyphrases': all_keyphrases,
            'ontology_mapped': resolved_entities,
            'processing_stats': {
                'total_keyphrases': len(all_keyphrases),
                'mapped_entities': sum(len(v) for v in resolved_entities.values())
            }
        }

def save_jsonl(data, filename):
    """Save data (list or dict) as JSONL file"""
    with open(filename, 'w', encoding='utf-8') as f:
        if isinstance(data, list):
            for item in data:
                f.write(json.dumps(item, ensure_ascii=False) + '\n')
        elif isinstance(data, dict):
            for key, value in data.items():
                f.write(json.dumps({'key': key, 'value': value}, ensure_ascii=False) + '\n')
        else:
            raise TypeError("Data must be a list or dict to save as JSONL.")

def create_summary(results):
    """Create summary statistics"""
    summary = {
        'total_chunks': len(results),
        'total_entities': 0,
        'entities_by_category': defaultdict(int),
        'most_common_entities': defaultdict(int)
    }

    all_entities = []
    for result in results:
        for category, entities in result['ontology_mapped'].items():
            summary['entities_by_category'][category] += len(entities)
            summary['total_entities'] += len(entities)
            all_entities.extend(entities)

    entity_counts = Counter(all_entities)
    summary['most_common_entities'] = dict(entity_counts.most_common(20))
    return summary

def main():
    ner_processor = NutritionNER()

    # Load chunks
    try:
        with open('data/raw_text.jsonl', 'r', encoding='utf-8') as f:
            chunks = [json.loads(line) for line in f]
    except FileNotFoundError:
        print("Error: raw_text.jsonl not found. Please run step 1 first.")
        return

    results = []
    print("Processing text chunks...")

    for i, chunk in enumerate(chunks):
        if i % 10 == 0:
            print(f"Processed {i}/{len(chunks)} chunks...")
        results.append(ner_processor.process_text_chunk(chunk))

    # Save as JSONL
    save_jsonl(results, 'ner_results.jsonl')

    # Create and save summary
    summary = create_summary(results)
    save_jsonl([summary], 'ner_summary.jsonl')

    print("\n✅ Processing complete!")
    print(f"Processed {len(results)} chunks")
    print(f"Total entities found: {summary['total_entities']}")
    print("\nEntities by category:")
    for category, count in summary['entities_by_category'].items():
        print(f"  {category}: {count}")

if __name__ == "__main__":
    main()


Processing text chunks...
Processed 0/134 chunks...
Processed 10/134 chunks...


In [ ]:
import spacy
import yake
from keybert import KeyBERT
import json
import pandas as pd
from collections import defaultdict
import re

class EnhancedNutritionNER:
    def __init__(self):
        # Load spaCy model
        self.nlp = spacy.load("en_core_web_sm")
        
        # Initialize YAKE
        self.yake_extractor = yake.KeywordExtractor(
            lan="en", 
            n=3,
            dedupLim=0.9,
            top=25
        )
        
        # Initialize KeyBERT
        self.keybert_model = KeyBERT()
        
        # Ontology patterns
        self.ontology_patterns = {
            'ingredient': [
                r'\b(?:whole grain|rye bread|oatmeal|barley|salmon|trout|herring|'
                r'beans|lentils|peas|soy|tofu|nuts|seeds|vegetables|fruits|berries|'
                r'potatoes|dairy|milk|cheese|yogurt|eggs|meat|poultry|fish|'
                r'cereals?|legumes?|root vegetables|leafy vegetables|cruciferous vegetables|'
                r'red meat|processed meat|poultry meat|seafood|shellfish)\b',
                r'\b(?:[A-Z][a-z]+ (?:oil|bread|milk|cheese|yogurt|fish|meat|drink|beverage))\b'
            ],
            'nutrient': [
                r'\b(?:vitamin [A-Z]|[A-Z] vitamins?|minerals?|iron|zinc|calcium|'
                r'selenium|iodine|fibre|fiber|protein|carbohydrates?|fat|sugar|'
                r'saturated fat|unsaturated fat|omega-3|omega-6|folate|'
                r'vitamin C|vitamin D|vitamin E|vitamin B12|vitamin B6|'
                r'sodium|salt|potassium|magnesium|phosphorus|copper|manganese)\b'
            ],
            'technique': [
                r'\b(?:cooking|boiling|steaming|frying|baking|roasting|grilling|'
                r'fermentation|soaking|germination|processing|preparation|'
                r'filtered|unfiltered|fortification|pasteurization|'
                r'food preparation|meal planning|menu planning)\b'
            ],
            'healthOutcome': [
                r'\b(?:cardiovascular disease|diabetes|obesity|cancer|'
                r'colorectal cancer|osteoporosis|anaemia|anemia|hypertension|'
                r'high blood pressure|stroke|dental caries|tooth decay|'
                r'metabolic syndrome|inflammation|premature mortality|'
                r'memory disorders|functional capacity|cognitive functions?)\b'
            ],
            'environmentImpact': [
                r'\b(?:climate impact|carbon footprint|greenhouse gas|'
                r'biodiversity loss|eutrophication|water footprint|'
                r'land use|resource use|environmental impact|'
                r'climate change|water scarcity|resource consumption)\b'
            ],
            'dietaryGuideline': [
                r'\b(?:nutrition recommendations?|dietary guidelines?|'
                r'food pyramid|plate model|recommended intake|'
                r'daily consumption|weekly consumption|'
                r'food-based dietary guidelines?|portion size|'
                r'meal pattern|eating pattern|diet composition)\b'
            ]
        }
        
        # Category keywords for fallback classification
        self.category_keywords = {
            'ingredient': ['grain', 'vegetable', 'fruit', 'meat', 'fish', 'dairy', 'nut', 'seed', 'bean'],
            'nutrient': ['vitamin', 'mineral', 'protein', 'fat', 'carbohydrate', 'fibre', 'calcium'],
            'technique': ['cook', 'boil', 'steam', 'fry', 'bake', 'process', 'prepare'],
            'healthOutcome': ['disease', 'cancer', 'diabetes', 'obesity', 'health risk'],
            'environmentImpact': ['climate', 'environment', 'biodiversity', 'footprint', 'impact'],
            'dietaryGuideline': ['recommend', 'guideline', 'advise', 'suggest', 'intake', 'consumption']
        }

    def extract_entities_spacy(self, text):
        doc = self.nlp(text)
        return [{'text': ent.text, 'label': ent.label_, 'start': ent.start_char, 'end': ent.end_char} for ent in doc.ents]

    def extract_keyphrases_yake(self, text):
        return [(kw[0], kw[1]) for kw in self.yake_extractor.extract_keywords(text)]

    def extract_keyphrases_keybert(self, text):
        return self.keybert_model.extract_keywords(
            text, keyphrase_ngram_range=(1, 3), stop_words='english', top_n=15
        )

    def classify_entity_fallback(self, phrase):
        phrase_lower = phrase.lower()
        category_scores = {}
        for category, keywords in self.category_keywords.items():
            score = sum(1 for keyword in keywords if keyword in phrase_lower)
            if score > 0:
                category_scores[category] = score
        return max(category_scores.items(), key=lambda x: x[1])[0] if category_scores else None

    def find_all_occurrences(self, text, phrase):
        occurrences = []
        pattern = re.compile(re.escape(phrase), re.IGNORECASE)
        for match in pattern.finditer(text):
            occurrences.append({
                'start': match.start(),
                'end': match.end(),
                'context': text[max(0, match.start()-50):match.end()+50]
            })
        return occurrences

    def map_to_ontology_enhanced(self, text, keyphrases):
        mapped_entities = defaultdict(list)
        all_phrases = []

        for phrase, score in keyphrases:
            all_phrases.append((phrase, score))
        unique_phrases = {}
        for phrase, score in all_phrases:
            if phrase not in unique_phrases or score < unique_phrases[phrase]:
                unique_phrases[phrase] = score

        for phrase, score in unique_phrases.items():
            classified = False
            for category, patterns in self.ontology_patterns.items():
                for pattern in patterns:
                    if re.search(pattern, phrase, re.IGNORECASE):
                        occurrences = self.find_all_occurrences(text, phrase)
                        if occurrences:
                            mapped_entities[category].append({
                                'entity': phrase,
                                'score': score,
                                'occurrences': occurrences
                            })
                        classified = True
                        break
                if classified:
                    break

            if not classified and len(phrase.split()) <= 3:
                fallback_category = self.classify_entity_fallback(phrase)
                if fallback_category:
                    occurrences = self.find_all_occurrences(text, phrase)
                    if occurrences:
                        mapped_entities[fallback_category].append({
                            'entity': phrase,
                            'score': score,
                            'occurrences': occurrences,
                            'classification_method': 'fallback'
                        })

        return dict(mapped_entities)

    def resolve_coreferences(self, entities):
        coreference_map = {
            'vitamin c': ['ascorbic acid', 'vit c', 'vitamin c'],
            'vitamin d': ['vit d', 'cholecalciferol', 'vitamin d'],
            'vitamin e': ['vit e', 'tocopherol', 'vitamin e'],
            'vitamin b12': ['vitamin b12', 'cobalamin', 'b12'],
            'fiber': ['fibre', 'dietary fiber', 'dietary fibre'],
            'saturated fat': ['saturated fatty acids', 'solid fat'],
            'unsaturated fat': ['unsaturated fatty acids', 'liquid fat'],
            'cardiovascular disease': ['cvd', 'heart disease'],
            'type 2 diabetes': ['t2d', 'adult-onset diabetes'],
            'whole grains': ['whole grain products', 'whole grain cereals'],
            'red meat': ['beef', 'pork', 'lamb'],
            'processed meat': ['sausages', 'cold cuts', 'salami']
        }
        resolved_entities = defaultdict(list)
        for category, entity_data in entities.items():
            for data in entity_data:
                entity_text = data['entity'].lower()
                resolved_text = entity_text
                for canonical, variants in coreference_map.items():
                    if entity_text in variants or canonical in entity_text:
                        resolved_text = canonical
                        break
                data['entity'] = resolved_text
                resolved_entities[category].append(data)
        return dict(resolved_entities)

    def process_text_chunk(self, chunk):
        text = chunk.get('text', '')
        metadata = chunk.get('metadata', {})
        spacy_entities = self.extract_entities_spacy(text)
        yake_keyphrases = self.extract_keyphrases_yake(text)
        keybert_keyphrases = self.extract_keyphrases_keybert(text)
        all_keyphrases = []
        for phrase, score in yake_keyphrases:
            all_keyphrases.append((phrase, score))
        for phrase, score in keybert_keyphrases:
            all_keyphrases.append((phrase, 1 - score))
        mapped_entities = self.map_to_ontology_enhanced(text, all_keyphrases)
        resolved_entities = self.resolve_coreferences(mapped_entities)
        return {
            'chunk_id': metadata.get('chunk_id', ''),
            'page': metadata.get('page', ''),
            'section': metadata.get('section', ''),
            'text_preview': text[:100] + '...' if len(text) > 100 else text,
            'spacy_entities': spacy_entities,
            'keyphrases': [phrase for phrase, score in all_keyphrases],
            'ontology_mapped': resolved_entities,
            'processing_stats': {
                'total_keyphrases': len(all_keyphrases),
                'mapped_entities': sum(len(v) for v in resolved_entities.values()),
                'total_occurrences': sum(sum(len(data['occurrences']) for data in v) for v in resolved_entities.values())
            }
        }

def create_comprehensive_entity_index(results):
    entity_index = defaultdict(lambda: {'class': None, 'occurrences': []})
    for chunk_result in results:
        chunk_id = chunk_result['chunk_id']
        page = chunk_result['page']
        section = chunk_result['section']
        for category, entity_data in chunk_result['ontology_mapped'].items():
            for data in entity_data:
                entity_name = data['entity']
                if entity_index[entity_name]['class'] is None:
                    entity_index[entity_name]['class'] = category
                elif entity_index[entity_name]['class'] != category:
                    entity_index[entity_name]['class'] += f", {category}"
                for occurrence in data['occurrences']:
                    entity_index[entity_name]['occurrences'].append({
                        'chunk_id': chunk_id,
                        'page': page,
                        'section': section,
                        'start_position': occurrence['start'],
                        'end_position': occurrence['end'],
                        'context': occurrence['context'],
                        'extraction_score': data.get('score', 0),
                        'classification_method': data.get('classification_method', 'pattern')
                    })
    return dict(sorted(entity_index.items(), key=lambda x: len(x[1]['occurrences']), reverse=True))

def create_summary(results, entity_index):
    summary = {
        'total_chunks': len(results),
        'total_unique_entities': len(entity_index),
        'total_occurrences': sum(len(data['occurrences']) for data in entity_index.values()),
        'entities_by_category': defaultdict(int),
        'most_common_entities': {},
        'classification_methods': defaultdict(int)
    }

    for entity_data in entity_index.values():
        categories = entity_data['class'].split(', ')
        for category in categories:
            summary['entities_by_category'][category] += 1
        for occurrence in entity_data['occurrences']:
            summary['classification_methods'][occurrence.get('classification_method', 'pattern')] += 1

    entity_freq = {entity: len(data['occurrences']) for entity, data in entity_index.items()}
    summary['most_common_entities'] = dict(sorted(entity_freq.items(), key=lambda x: x[1], reverse=True)[:20])
    return summary

def save_jsonl(data, filename):
    """Helper to save list or dict as JSONL."""
    with open(filename, 'w', encoding='utf-8') as f:
        if isinstance(data, list):
            for item in data:
                f.write(json.dumps(item, ensure_ascii=False) + '\n')
        elif isinstance(data, dict):
            for key, value in data.items():
                f.write(json.dumps({'key': key, 'value': value}, ensure_ascii=False) + '\n')
        else:
            raise TypeError("Data must be a list or dict to save as JSONL.")

def main():
    ner_processor = EnhancedNutritionNER()
    try:
        with open('data/raw_text.jsonl', 'r', encoding='utf-8') as f:
            chunks = [json.loads(line) for line in f]
    except FileNotFoundError:
        print("Error: raw_text.jsonl not found. Please run step 1 first.")
        return

    results = []
    print("Processing text chunks with enhanced NER...")
    for i, chunk in enumerate(chunks):
        if i % 10 == 0:
            print(f"Processed {i}/{len(chunks)} chunks...")
        results.append(ner_processor.process_text_chunk(chunk))

    # Save results
    save_jsonl(results, 'ner_results_enhanced.jsonl')
    entity_index = create_comprehensive_entity_index(results)
    save_jsonl(entity_index, 'entity_index.jsonl')
    summary = create_summary(results, entity_index)
    save_jsonl([summary], 'ner_summary_enhanced.jsonl')

    print("\nProcessing complete!")
    print(f"Processed {len(results)} chunks")
    print(f"Total unique entities: {summary['total_unique_entities']}")
    print(f"Total entity occurrences: {summary['total_occurrences']}")
    print("\nEntities by category:")
    for category, count in summary['entities_by_category'].items():
        print(f"  {category}: {count} entities")

if __name__ == "__main__":
    main()


Processing text chunks with enhanced NER...
Processed 0/134 chunks...
Processed 10/134 chunks...
Processed 20/134 chunks...
Processed 30/134 chunks...
Processed 40/134 chunks...
Processed 50/134 chunks...
Processed 60/134 chunks...
Processed 70/134 chunks...
Processed 80/134 chunks...
Processed 90/134 chunks...
Processed 100/134 chunks...
Processed 110/134 chunks...
Processed 120/134 chunks...
Processed 130/134 chunks...

Processing complete!
Processed 134 chunks
Total unique entities: 762
Total entity occurrences: 4169

Entities by category:
  dietaryGuideline: 138 entities
  ingredient: 325 entities
  nutrient: 207 entities
  environmentImpact: 54 entities
  technique: 24 entities
  healthOutcome: 16 entities
